# Dijet JER systematic uncertainty

Compare the JER Up, Down, and default reconstructed dijet pseudorapidity distributions in embedding or Pythia MC. Full CM distributions are normalized to unit integral before comparison. Forward/Backward distributions are left unnormalized; their ratios always use standard independent-error propagation, never ROOT's binomial option.

Dedicated variation/default plots provide the signed shape variations used to estimate the JER systematic uncertainty.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace
from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import COLORS, DEFAULT_PLOT_STYLE

## Configuration

`FULL_COMPARISON_RATIO_OPTION` controls errors on normalized Up/Def and Down/Def shape ratios. `FB_COMPARISON_RATIO_OPTION` controls errors only on the later ratio of two already-constructed F/B histograms. Set either to `''` for ROOT's standard propagation or `'B'` for option B. The construction of every F/B histogram is hard-coded to `''`.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS = tuple(DIJET_PTAVE_BINS)
REBIN_ETA = 2
FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: F / B is never binomial
FULL_COMPARISON_RATIO_OPTION = ''   # '' or 'B' for Up/Def and Down/Def
FB_COMPARISON_RATIO_OPTION = ''     # '' or 'B' for (F/B)_var / (F/B)_Def
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_JER_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_JER',
))

CURVES = (
    DijetClosureCurve(
        'JER Up', 'hRecoDijetPtEtaCMJerUp_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerUp_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerUp_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JER Down', 'hRecoDijetPtEtaCMJerDown_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerDown_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerDown_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JER Default', 'hRecoDijetPtEtaCMJerDef_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerDef_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerDef_{eta_cut_index}',
    ),
)
NOMINAL = 'JER Default'
# Select entries from root_style.COLORS (0 red, 1 blue, 2 black, ...).
HISTOGRAM_COLOR_INDICES = {'JER Up': 0, 'JER Down': 1, 'JER Def.': 2}
VARIATION_COLOR_INDICES = {'Up / Def.': 0, 'Down / Def.': 1}
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026, annotation_line_spacing=0.039,
    legend_text_size=0.028,
)
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError('GENERATOR must be embedding or pythia')
if DIRECTION not in ('pgoing', 'Pbgoing', 'combined'):
    raise ValueError('DIRECTION must be pgoing, Pbgoing, or combined')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('Forward/Backward construction must use standard errors')
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
for label, color_index in (
    *HISTOGRAM_COLOR_INDICES.items(), *VARIATION_COLOR_INDICES.items(),
):
    if not isinstance(color_index, int) or not 0 <= color_index < len(COLORS):
        raise ValueError(f'Invalid root_style color index for {label}: {color_index}')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]

In [ ]:
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

DIRECTION_LABELS = {
    'pgoing': 'p-going', 'Pbgoing': 'Pb-going', 'combined': 'combined',
}
INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured MC ROOT file: {INPUT_FILE}')
INPUT_FILE

## Build projections and systematic comparisons

Each CM projection is scaled by `1 / Integral()` through `normalization='integral'`. Forward and backward projections are not normalized before division.

In [ ]:
jer_results = {}
eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
eta_cut_tag = int(round(10.0 * ETA_CUT))

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    common_tag = (
        f'{GENERATOR}_{DIRECTION}_jerSystematics_etaCM_{eta_cut_tag}'
        f'_ptave_{ptave_tag}'
    )
    output_name = lambda plot: (
        f'{GENERATOR}_{DIRECTION}_jerSystematics_{plot}'
        f'_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf'
    )
    eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
        INPUT_FILE, CURVES, eta_cut_index=ETA_CUT_INDEX,
        ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
        normalization='integral',
        ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
    )
    eta_variations = {
        ratio_label: ratio_to_nominal(
            eta_shapes[source_label], eta_shapes[NOMINAL],
            name=f'h_{common_tag}_{source_label.replace(" ", "_")}_to_default',
            option=FULL_COMPARISON_RATIO_OPTION,
        )
        for ratio_label, source_label in (
            ('Up / Def', 'JER Up'), ('Down / Def', 'JER Down'),
        )
    }
    fb_variations = {
        ratio_label: ratio_to_nominal(
            fb_ratios[source_label], fb_ratios[NOMINAL],
            name=f'h_{common_tag}_{source_label.replace(" ", "_")}_fb_to_default',
            option=FB_COMPARISON_RATIO_OPTION,
        )
        for ratio_label, source_label in (
            ('Up / Def', 'JER Up'), ('Down / Def', 'JER Down'),
        )
    }
    annotations = (
        GENERATOR.capitalize(),
        DIRECTION_LABELS[DIRECTION],
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} > 50 GeV',
        'p_{T}^{SubLead} > 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    canvases = {
        'eta_overlay': draw_overlay(
            eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
            y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('full_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay',
        ),
        'eta_variations': draw_overlay(
            eta_variations, title='', x_title='#eta_{CM}^{dijet}',
            y_title='JER variation / default', x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('full_ratio_to_default'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_ratio',
        ),
        'fb_overlay': draw_overlay(
            fb_ratios, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Forward / Backward', x_range=fb_x_range,
            y_range=FB_RANGE, annotations=annotations, grid=DRAW_GRID,
            style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('fb_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay',
        ),
        'fb_variations': draw_overlay(
            fb_variations, title='', x_title='#eta_{CM}^{dijet}',
            y_title='(F/B)_{JER variation} / (F/B)_{default}',
            x_range=fb_x_range, y_range=FB_DOUBLE_RATIO_RANGE,
            reference_y=1.0, annotations=annotations, grid=DRAW_GRID,
            style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('fb_ratio_to_default'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_ratio',
        ),
    }
    jer_results[ptave_range] = {
        'eta_shapes': eta_shapes, 'eta_variations': eta_variations,
        'forward_backward': fb_ratios, 'fb_variations': fb_variations,
        'keys': selected_keys, 'canvases': canvases,
    }
    print(ptave_range, selected_keys)
    for canvas in canvases.values():
        display(canvas)

## Validation summary

Check the unit-integral normalization and report the finite extrema of the signed systematic ratios. The envelope per bin can be read as `max(abs(Up/Def - 1), abs(Down/Def - 1))`.

In [ ]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None

for ptave_range, result in jer_results.items():
    print(f'\npTave interval {ptave_range}, eta cut {ETA_CUT:g}')
    print('CM integral normalization:', {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    })
    print('CM variation/default ranges:', {
        label: finite_nonzero_range(histogram)
        for label, histogram in result['eta_variations'].items()
    })
    print('F/B variation/default ranges:', {
        label: finite_nonzero_range(histogram)
        for label, histogram in result['fb_variations'].items()
    })